# Modeling Analysis: LIAR Binary Credibility Prediction

This notebook trains and evaluates lightweight supervised models for binary credibility classification on the processed LIAR dataset. LIAR is used because it provides short political claims with PolitiFact truthfulness labels and fixed train/validation/test splits. Data is processed in an upstream exploration notebook [data_exploration](data_exploration.ipynb)

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('.').resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from models import (
    DummyBaselineModel,
    TfidfLogisticRegressionModel,
    SentenceEmbeddingLogisticRegressionModel,
    SentenceEmbeddingGBMModel,
)

In [2]:
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'liar_binary.csv'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'

for path in [MODELS_DIR, REPORTS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

Use the original LIAR train/valid/test split to avoid accidental data leakage between training and evaluation.

In [3]:
df = pd.read_csv(DATA_PATH)

train_df = df[df['split'] == 'train'].copy()
valid_df = df[df['split'] == 'valid'].copy()
test_df = df[df['split'] == 'test'].copy()

X_train = train_df['statement'].fillna('')
y_train = train_df['label']

X_valid = valid_df['statement'].fillna('')
y_valid = valid_df['label']

X_test = test_df['statement'].fillna('')
y_test = test_df['label']

label_names = ['not_credible', 'credible']

In [4]:
def evaluate_binary_model(name: str, y_true, y_pred) -> dict[str, float | str]:
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average='binary',
        pos_label=1,
        zero_division=0,
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average='macro',
        zero_division=0,
    )
    return {
        'model': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_credible': precision,
        'recall_credible': recall,
        'f1_credible': f1,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
    }

In [5]:
def save_confusion_matrix(y_true, y_pred, title: str, output_path: Path) -> None:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    display.plot(values_format='d')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    print(f"Figure saved to: {str(output_path.relative_to(PROJECT_ROOT))}")
    plt.close()

In [6]:
metrics_rows = []
prediction_frames = []

## Dummy Model Baseline

In [7]:
dummy = DummyBaselineModel()
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

In [8]:
metrics_rows.append(evaluate_binary_model('Dummy baseline', y_test, dummy_pred))
print(classification_report(y_test, dummy_pred, target_names=label_names, zero_division=0))

save_confusion_matrix(
    y_test,
    dummy_pred,
    'Dummy baseline confusion matrix',
    FIGURES_DIR / 'confusion_matrix_dummy.png',
)

              precision    recall  f1-score   support

not_credible       0.55      1.00      0.71       504
    credible       0.00      0.00      0.00       410

    accuracy                           0.55       914
   macro avg       0.28      0.50      0.36       914
weighted avg       0.30      0.55      0.39       914

Figure saved to: reports/figures/confusion_matrix_dummy.png


![Chart](reports/figures/confusion_matrix_dummy.png)

In [9]:
dummy.save(MODELS_DIR / 'dummy_baseline.joblib')
print(f"Sanity check! Dummy model file was succesfully saved: {(MODELS_DIR / 'dummy_baseline.joblib').exists()}")

Sanity check! Dummy model file was succesfully saved: True


The majority-class dummy model is a minimum reference point and helps verify whether later models learn any claim-level signal beyond class imbalance.

## TF-IDF Model

TF-IDF converts each statement into weighted word and phrase features. Logistic Regression learns which phrases are more associated with credible or not credible statements. For a given prediction, the most influential phrases are approximated by coefficient x TF-IDF value.

In [10]:
model_tfidf = TfidfLogisticRegressionModel()
model_tfidf.fit(X_train, y_train)
tfidf_pred = model_tfidf.predict(X_test)

In [11]:
metrics_rows.append(evaluate_binary_model("TF-IDF + Logistic Regression", y_test, tfidf_pred))
print(classification_report(y_test, tfidf_pred, target_names=label_names, zero_division=0))

save_confusion_matrix(
    y_test,
    tfidf_pred,
    "TF-IDF Logistic Regression confusion matrix",
    FIGURES_DIR / "confusion_matrix_tfidf.png",
)

              precision    recall  f1-score   support

not_credible       0.66      0.60      0.63       504
    credible       0.56      0.62      0.59       410

    accuracy                           0.61       914
   macro avg       0.61      0.61      0.61       914
weighted avg       0.62      0.61      0.61       914

Figure saved to: reports/figures/confusion_matrix_tfidf.png


![Chart](reports/figures/confusion_matrix_tfidf.png)

In [12]:
model_tfidf.save(MODELS_DIR / "tfidf_logreg.joblib")
print(f"Sanity check! TF-IDF model file was succesfully saved: {(MODELS_DIR / "tfidf_logreg.joblib").exists()}")

Sanity check! TF-IDF model file was succesfully saved: True


In [13]:
vectorizer = model_tfidf.named_steps["tfidf"]
clf = model_tfidf.named_steps["clf"]
feature_names = vectorizer.get_feature_names_out()
coefs = clf.coef_[0]

coef_df = pd.DataFrame({
    "ngram": feature_names,
    "coefficient": coefs,
})

credible_terms = (
    coef_df.sort_values("coefficient", ascending=False)
    .head(20)
    .assign(direction="credible")
)
not_credible_terms = (
    coef_df.sort_values("coefficient", ascending=True)
    .head(20)
    .assign(direction="not_credible")
)

top_ngrams = pd.concat([credible_terms, not_credible_terms], ignore_index=True)
top_ngrams.to_csv(REPORTS_DIR / "tfidf_top_ngrams.csv", index=False)
top_ngrams

,ngram,coefficient,direction
0,percent,2.481705,credible
1,georgia,1.759950,credible
2,day,1.743449,credible
3,average,1.717966,credible
4,half,1.688768,credible
5,months,1.681970,credible
6,times,1.667909,credible
7,terms,1.500619,credible
8,american,1.432239,credible
9,debt,1.398149,credible


In [14]:
def explain_tfidf_prediction(text: str, top_n: int = 10) -> pd.DataFrame:
    transformed = vectorizer.transform([text])
    row = transformed.tocoo()
    contributions = []
    for _, feature_idx, tfidf_value in zip(row.row, row.col, row.data):
        contributions.append({
            "ngram": feature_names[feature_idx],
            "tfidf_value": tfidf_value,
            "coefficient": coefs[feature_idx],
            "contribution": tfidf_value * coefs[feature_idx],
        })
    return (
        pd.DataFrame(contributions)
        .sort_values("contribution", key=lambda s: s.abs(), ascending=False)
        .head(top_n)
    )

example_indices = test_df.sample(3, random_state=42).index
explanation_rows = []

for idx in example_indices:
    text = test_df.loc[idx, "statement"]
    pred = int(model_tfidf.predict([text])[0])
    explanation = explain_tfidf_prediction(text, top_n=8)
    for _, row in explanation.iterrows():
        explanation_rows.append({
            "statement_id": test_df.loc[idx, "statement_id"],
            "statement": text,
            "true_label": int(test_df.loc[idx, "label"]),
            "predicted_label": pred,
            **row.to_dict(),
        })

pd.DataFrame(explanation_rows).to_csv(REPORTS_DIR / "tfidf_example_explanations.csv", index=False)

Explanations of TF-IDF model predictions can be read at: [reports/tfidf_example_explanations.csv](reports/tfidf_example_explanations.csv)

## Sentence-Transformers with Logistic Regression Model

This model uses a pretrained sentence-transformer to convert each short claim into dense semantic embeddings. A lightweight Logistic Regression classifier is trained on top of those embeddings.

In [ ]:
model_embeddings = SentenceEmbeddingLogisticRegressionModel()
model_embeddings.fit(X_train, y_train)
embedding_pred = model_embeddings.predict(X_test)

In [16]:
model_embeddings.save(MODELS_DIR / "embedding_logreg.joblib")
print(f"Sanity check! Embeddings Logreg model file was succesfully saved: {(MODELS_DIR / "embedding_logreg.joblib").exists()}")

Sanity check! Embeddings Logreg model file was succesfully saved: True


In [17]:
metrics_rows.append(evaluate_binary_model("Sentence embeddings + Logistic Regression", y_test, embedding_pred))
print(classification_report(y_test, embedding_pred, target_names=label_names, zero_division=0))

save_confusion_matrix(
    y_test,
    embedding_pred,
    "Sentence embeddings confusion matrix",
    FIGURES_DIR / "confusion_matrix_embeddings.png",
)

              precision    recall  f1-score   support

not_credible       0.67      0.61      0.64       504
    credible       0.57      0.63      0.60       410

    accuracy                           0.62       914
   macro avg       0.62      0.62      0.62       914
weighted avg       0.62      0.62      0.62       914

Figure saved to: reports/figures/confusion_matrix_embeddings.png


![Chart](reports/figures/confusion_matrix_embeddings.png)

## Sentence-Transformers with GBM Model

This model uses a pretrained sentence-transformer to convert each short claim into dense semantic embeddings. A Gradient Boosting Classifier is trained on top of those embeddings.

In [ ]:
model_embeddings_gbm = SentenceEmbeddingGBMModel()
model_embeddings_gbm.fit(X_train, y_train)
gbm_embedding_pred = model_embeddings_gbm.predict(X_test)

In [19]:
model_embeddings_gbm.save(MODELS_DIR / "embedding_gbm.joblib")
print(f"Sanity check! Embeddings GBM model file was succesfully saved: {(MODELS_DIR / "embedding_gbm.joblib").exists()}")

Sanity check! Embeddings GBM model file was succesfully saved: True


In [20]:
metrics_rows.append(evaluate_binary_model("Sentence embeddings + GBM", y_test, gbm_embedding_pred))
print(classification_report(y_test, gbm_embedding_pred, target_names=label_names, zero_division=0))

              precision    recall  f1-score   support

not_credible       0.66      0.65      0.66       504
    credible       0.58      0.60      0.59       410

    accuracy                           0.63       914
   macro avg       0.62      0.62      0.62       914
weighted avg       0.63      0.63      0.63       914



In [21]:
save_confusion_matrix(
    y_test,
    gbm_embedding_pred,
    "Sentence embeddings with GBM confusion matrix",
    FIGURES_DIR / "confusion_matrix_embeddings_gbm.png",
)

Figure saved to: reports/figures/confusion_matrix_embeddings_gbm.png


![Chart](reports/figures/confusion_matrix_embeddings_gbm.png)

## Modeling limitations

The dummy baseline is a minimum reference point. TF-IDF + Logistic Regression is the main explainable supervised baseline. Sentence embeddings + Logistic Regression tests pretrained semantic signal without fine-tuning, and Sentence embeddings + GBM tests a nonlinear classifier on the same embedding space. The main limitations are the short-claim dataset format, lost nuance from binary labeling, political-domain bias, and the absence of evidence retrieval or real-time fact checking.

## Final model comparison

Macro F1 is the main comparison metric because the binary classes are not perfectly balanced, so accuracy can hide weak performance on the smaller credible class. The dummy baseline is the minimum reference point. TF-IDF should beat that baseline if the statement text contains repeatable signal. Sentence embeddings test whether pretrained semantic representations add useful signal beyond sparse lexical features. The embedding GBM tests a nonlinear classifier on the same semantic representation.

In [22]:
model_values = {
    "Dummy baseline": "minimum reference",
    "TF-IDF + Logistic Regression": "strong explainable supervised baseline",
    "Sentence embeddings + Logistic Regression": "semantic transformer-based signal without fine-tuning",
    "Sentence embeddings + GBM": "semantic transformer-based signal with GBM",
}

metrics_summary = pd.DataFrame(metrics_rows)
metrics_summary["main_value"] = metrics_summary["model"].map(model_values)
metrics_summary = metrics_summary.sort_values("macro_f1", ascending=False)
metrics_summary.to_csv(REPORTS_DIR / "metrics_summary.csv", index=False)
metrics_summary

,model,accuracy,precision_credible,recall_credible,f1_credible,macro_precision,macro_recall,macro_f1,main_value
3,Sentence embeddings + GBM,0.626915,0.582339,0.595122,0.588661,0.623493,0.623950,0.623660,semantic transformer-based signal with GBM
2,Sentence embeddings + Logistic Regression,0.617068,0.565789,0.629268,0.595843,0.616956,0.618206,0.616009,semantic transformer-based signal without fine...
1,TF-IDF + Logistic Regression,0.610503,0.558952,0.624390,0.589862,0.610616,0.611798,0.609514,strong explainable supervised baseline
0,Dummy baseline,0.551422,0.000000,0.000000,0.000000,0.275711,0.500000,0.355430,minimum reference


## Error analysis

The following section compares model predictions on the held-out LIAR test split. A false positive means a not-credible statement was predicted as credible. A false negative means a credible statement was predicted as not credible. The review focuses on TF-IDF because it is the strongest explainable baseline, then compares it with the sentence-embedding GBM.

In [23]:
error_df = test_df[[
    "statement_id",
    "original_label",
    "label",
    "label_name",
    "statement",
    "subject",
    "speaker",
    "party_affiliation",
    "context",
    "text_length_words",
]].copy()

error_df["pred_dummy"] = dummy_pred
error_df["pred_tfidf"] = tfidf_pred
error_df["pred_embeddings"] = embedding_pred
error_df["pred_embeddings_gbm"] = gbm_embedding_pred

tfidf_false_positives = error_df[(error_df["label"] == 0) & (error_df["pred_tfidf"] == 1)].head(10)
tfidf_false_negatives = error_df[(error_df["label"] == 1) & (error_df["pred_tfidf"] == 0)].head(10)

comparison_patterns = {
    "both_correct": int(((error_df["pred_tfidf"] == error_df["label"]) & (error_df["pred_embeddings_gbm"] == error_df["label"])).sum()),
    "tfidf_correct_embeddings_gbm_wrong": int(((error_df["pred_tfidf"] == error_df["label"]) & (error_df["pred_embeddings_gbm"] != error_df["label"])).sum()),
    "tfidf_wrong_embeddings_gbm_correct": int(((error_df["pred_tfidf"] != error_df["label"]) & (error_df["pred_embeddings_gbm"] == error_df["label"])).sum()),
    "both_wrong": int(((error_df["pred_tfidf"] != error_df["label"]) & (error_df["pred_embeddings_gbm"] != error_df["label"])).sum()),
}

comparison_patterns

{'both_correct': 425,
 'tfidf_correct_embeddings_gbm_wrong': 133,
 'tfidf_wrong_embeddings_gbm_correct': 148,
 'both_wrong': 208}

In [24]:
tfidf_false_positives[["original_label", "statement", "subject", "speaker", "context", "text_length_words"]]

,original_label,statement,subject,speaker,context,text_length_words
8297,false,wisconsin pace double number layoffs year,jobs,katrina-shankland,a news conference,6
8307,false,says percent federal spending goes military pe...,"federal-budget,military,poverty",facebook-posts,a meme on social media,13
8313,pants-fire,number illegal immigrants could million could ...,immigration,donald-trump,"a speech in Phoenix, Ariz.",7
8319,false,rosemary lehmberg travis county das office con...,"crime,criminal-justice",sarah-palin,a commentary,12
8327,barely-true,says bag litter increased san francisco banned...,environment,drew-springer,a press release,10
8331,false,proposed tax fund transportation projects woul...,transportation,steve-brown,a forum hosted by The Atlanta Journal-Constitu...,14
8335,false,undocumented children picked border told appea...,"children,foreign-policy,immigration,legal-issues",jeff-flake,"an interview on the PBS ""News Hour""",10
8338,pants-fire,sixty percent hispanics support arizona immigr...,immigration,russell-pearce,a CNN interview,7
8348,barely-true,state tax law start house renewal state hospit...,"state-finances,taxes",david-pennington,a public meeting,14
8350,false,strong bipartisan majority house representativ...,"congressional-rules,federal-budget,health-care",ted-cruz,a news release,8


In [25]:
tfidf_false_negatives[["original_label", "statement", "subject", "speaker", "context", "text_length_words"]]

,original_label,statement,subject,speaker,context,text_length_words
8296,true,building wall usmexico border take literally y...,immigration,rick-perry,Radio interview,7
8310,true,year people die america dont health care,health-care,hillary-clinton,"a speech in Des Moines, Iowa.",7
8312,mostly-true,public safety issues cities allow transgender ...,"crime,gays-and-lesbians,sexuality",chris-sgro,a speech urging Charlotte's anti-discriminatio...,11
8315,true,time someone like scalia ginsburg got plus votes,"sotomayor-nomination,supreme-court",lindsey-graham,a Senate hearing,8
8316,true,contends president obama literally said capand...,"climate-change,energy",mike-pence,MSNBC interview.,14
8322,mostly-true,massachusetts scott brown pushed law force wom...,"abortion,legal-issues,women",jeanne-shaheen,an ad,15
8323,true,fed created trillion nothing gave banks foreig...,"economy,financial-regulation",dennis-kucinich,a television interview,11
8341,true,says bill hillary clinton attended donald trum...,candidates-biography,carlos-curbelo,a radio interview,9
8344,mostly-true,says marco rubio said social security medicare...,"medicare,social-security",patrick-murphy,a Florida Senate debate,10
8346,true,secretary department energy former president b...,energy,bill-richardson,"a debate in Manchester, N.H.",12


In [26]:
def _format_examples(rows: pd.DataFrame, limit: int = 5) -> str:
    examples = []
    for _, row in rows.head(limit).iterrows():
        statement = str(row["statement"]).replace("\n", " " )
        examples.append(
            f"- {row['original_label']} | {row['speaker']} | {row['subject']}: {statement}"
        )
    return "\n".join(examples)

best_model = metrics_summary.iloc[0]
tfidf_row = metrics_summary[metrics_summary["model"] == "TF-IDF + Logistic Regression"].iloc[0]
dummy_row = metrics_summary[metrics_summary["model"] == "Dummy baseline"].iloc[0]
embedding_row = metrics_summary[metrics_summary["model"] == "Sentence embeddings + Logistic Regression"].iloc[0]
gbm_row = metrics_summary[metrics_summary["model"] == "Sentence embeddings + GBM"].iloc[0]

error_report = f"""# Error Analysis

## 1. Most common false positive patterns

Manual review of 10 TF-IDF false positives suggests that not-credible statements are often predicted as credible when they contain concrete numeric claims, institutional language, or matter-of-fact policy wording. Several examples look plausible without the missing PolitiFact context, which is a hard setting for statement-only modeling.

{_format_examples(tfidf_false_positives)}

## 2. Most common false negative patterns

Manual review of 10 TF-IDF false negatives suggests that credible statements are often predicted as not credible when they use adversarial political framing, mention polarizing subjects, or resemble common campaign-attack phrasing. This is consistent with a lexical model picking up topic and rhetoric shortcuts.

{_format_examples(tfidf_false_negatives)}

## 3. Examples where the original label is debatable

Some errors are short, context-dependent claims where the truth label depends on date, jurisdiction, or a narrow definition. Those examples are difficult for statement-only models because the model cannot verify the external evidence behind the claim.

## 4. What TF-IDF learned well

TF-IDF improved macro F1 from {dummy_row['macro_f1']:.3f} for the dummy baseline to {tfidf_row['macro_f1']:.3f}. It learned repeatable lexical cues and remains the easiest model to inspect through top n-grams and per-statement feature contributions.

## 5. What the embedding model added

Sentence embeddings with Logistic Regression reached macro F1 {embedding_row['macro_f1']:.3f}. Compared with TF-IDF, this tests whether pretrained semantic similarity helps on short claims without fine-tuning. In this run it is best treated as a semantic baseline rather than a clear replacement for TF-IDF.

## 6. What Sentence embeddings + GBM helped explain

Sentence embeddings + GBM reached macro F1 {gbm_row['macro_f1']:.3f}. The TF-IDF-vs-GBM comparison was {comparison_patterns}. This shows whether a nonlinear model on semantic embeddings corrects a meaningful number of TF-IDF mistakes or mostly shifts the error profile.

## 7. Main limitations

1. LIAR contains short statements, not full articles.
2. Binary mapping loses nuance.
3. Dropping `half-true` makes the task cleaner but less complete.
4. Political-domain bias is likely.
5. No real-time fact checking is performed.
"""

(REPORTS_DIR / "error_analysis.md").write_text(error_report, encoding="utf-8")
print(f"Error analysis saved to: {str((REPORTS_DIR / "error_analysis.md").relative_to(PROJECT_ROOT))}")

Error analysis saved to: reports/error_analysis.md


Error analysis can be viewed here: [reports/error_analysis.md](reports/error_analysis.md)